<a href="https://colab.research.google.com/github/panashemuringa/MURINGA-PANASHE-P-HASTS-211-ASSIGNMENT/blob/main/Panashe_Priscilla_Muringa_Assignment_2_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Panashe Priscilla Muringa R2420866
### HASTS211 Assignment

##### Identifying and Detecting a Regime Change in the AAPL Stock Returns from yahoo finance

This notebook applies the Gaussian Mixture Model (GMM) to detect the change in the regime

Source of data: Yahoo Finance AAPL data
Dataset : AAPL data from 1 January 2018 to 31 December 2025

---
## 1. Definition

A **regime change** in a time series refers to the changing behaviour of the time series in different time intervals. Instead of assuming a single stable process, we allow the data to switch between $K$ different states (or regimes). There are several methods that can be applied to identify market regime. The technique chosen for this notebook is the **Gaussian Mixture Model (GMM)**.
#### Why the Gaussian Mixture Model
The GMM is well suited for regime detection because it can decompose a complex, multimodal return distribution into simpler Gaussian components, each representing a distinct market state. It is straightforward to fit, provides interpretable parameters, and allows soft probabilistic assignment of each day to a regime via posterior probabilities

A GMM represents the overall distribution of observations as a weighted combination of $K$ Gaussian (normal) distributions, each corresponding to a different regime. Unlike the Hidden Markov Model, the GMM treats each observation **independently** — it assigns each day's return to a component based purely on how well that return fits each Gaussian's shape, without regard to the previous day's regime.

The probability of observing a return $y$ is:

$$p(y) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(y \mid \mu_k,\, \sigma_k^2)$$

where:
- $\pi_k$ is the **mixing weight** of component $k$ (the proportion of days in that regime), with $\sum_k \pi_k = 1$
- $\mu_k$ is the **mean return** of component $k$
- $\sigma_k^2$ is the **variance** of returns in component $k$

## 2. Description

The Gaussian Mixture Model assumes that the stock return on any given day is drawn from one of two latent market components: a low-volatility component (bull market) or a high-volatility component (bear market or crisis). The model is fitted using the **Expectation-Maximisation (EM) algorithm**, which iterates between assigning each observation a soft probability of belonging to each component (E-step) and updating the component parameters to best explain those assignments (M-step).

Unlike the HMM, the GMM does not model the sequence or persistence of regimes — it partitions the return distribution into distinct clusters. This makes it simpler to fit and interpret, but it cannot capture the fact that bull markets tend to persist for months before switching to bear markets.

### Installing the necessary modules and packages

In this notebook plotly will be used instead of matplotlib so that we can be able to zoom in on our plots and reduce or increase the time (for example we can zoom in until we are able to see the daily plot rather than the fixed years.)

In [1]:
%pip install yfinance scikit-learn plotly statsmodels

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.graph_objs.scatter.marker import Line
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from statsmodels.tsa.stattools import adfuller
from scipy import stats
import warnings
import math

warnings.filterwarnings('ignore')

### Loading the Data
We are going to use yfinance module to download the finance data from yahoo finance for Apple Inc.



In [3]:
#Loading the Apple data
apple_data = yf.download("AAPL",start='2018-01-01',end='2025-12-31',auto_adjust=True)
apple = pd.DataFrame(apple_data)
apple.columnns = apple_data['Close']

#computing log_returns
apple['log_returns']= np.log(apple['Close']/apple['Close'].shift(1))
apple = apple.dropna()
print(f'Observations: {len(apple)}')
print(f'Dates : {apple.index[0].date()} to {apple.index[-1].date()}')
apple.head()

[*********************100%***********************]  1 of 1 completed

Observations: 2009
Dates : 2018-01-03 to 2025-12-30


Price,Close,High,Low,Open,Volume,log_returns
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,
Date,,,,,,
2018-01-03,40.297153,40.839972,40.233983,40.367346,118071600,-0.000174
2018-01-04,40.484333,40.587282,40.262059,40.369685,89738400,0.004634
2018-01-05,40.945271,41.031839,40.489024,40.580273,94640000,0.011321
2018-01-08,40.793171,41.087975,40.694899,40.793171,82271200,-0.003722
2018-01-09,40.788502,40.959301,40.573247,40.839976,86336000,-0.000114


The missing values must have been left out by the drop.na() functions, which means they would have produced null values which we should drop for proper data analysis.

---
## 3. Diagram — Exploratory Plots

Before fitting the model, it helps to look at the data and check whether there are visible periods of different behaviour.

In [4]:
#plot for the closing prices for each day
px.line(apple_data['Close'])

In [5]:
#plot for the log returns
px.line(apple['log_returns'], width=800, height=500, labels= {'Date','Log returns'})

In [6]:
# Rolling 30-day annualised volatility
rolling_vol = apple['log_returns'].rolling(30).std() * np.sqrt(252)

fig = make_subplots(rows=3, cols=1,
    subplot_titles=['AAPL Adjusted Close Price (2018–2025)',
                    'AAPL Daily Log Returns',
                    'AAPL 30-Day Rolling Annualised Volatility'],
    shared_xaxes=True, vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=apple.index, y=apple_data['Close'].squeeze(),
    line=dict(color='steelblue'), name='Close Price'), row=1, col=1)

fig.add_trace(go.Scatter(x=apple.index, y=apple['log_returns'],
    line=dict(color='darkorange', width=0.8), name='Log Return'), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=0.8, row=2, col=1)

fig.add_trace(go.Scatter(x=apple.index, y=rolling_vol,
    line=dict(color='crimson'), name='Rolling Vol'), row=3, col=1)

fig.update_yaxes(title_text='Price (USD)', row=1, col=1)
fig.update_yaxes(title_text='Log Return', row=2, col=1)
fig.update_yaxes(title_text='Ann. Volatility', row=3, col=1)
fig.update_layout(height=800, width=900, showlegend=False, margin=dict(l=20,r=20,t=60,b=20))
fig.show()
print('You can see clear spikes in volatility around 2020 (COVID-19 pandemic), most 0f 2022 (interest rate spikes) and 2025 (US administration changes)')

You can see clear spikes in volatility around 2020 (COVID-19 pandemic), most 0f 2022 (interest rate spikes) and 2025 (US administration changes)


In [7]:
# Return distribution: histogram vs normal + QQ plot
lr = apple['log_returns'].dropna()
mu, sigma = lr.mean(), lr.std()
x_range = np.linspace(lr.min(), lr.max(), 200)

# QQ data
(osm, osr), (slope, intercept, _) = stats.probplot(lr, dist='norm')
qq_line_x = np.array([osm[0], osm[-1]])
qq_line_y = intercept + slope * qq_line_x

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Return Distribution vs Normal', 'QQ Plot of AAPL Log Returns'])

fig.add_trace(go.Histogram(x=lr, nbinsx=70, histnorm='probability density',
    marker_color='steelblue', opacity=0.6, name='AAPL returns'), row=1, col=1)
fig.add_trace(go.Scatter(x=x_range, y=stats.norm.pdf(x_range, mu, sigma),
    line=dict(color='red', width=2), name='Normal'), row=1, col=1)

fig.add_trace(go.Scatter(x=osm, y=osr, mode='markers',
    marker=dict(color='steelblue', size=3), name='QQ'), row=1, col=2)
fig.add_trace(go.Scatter(x=qq_line_x, y=qq_line_y,
    line=dict(color='red'), name='QQ line'), row=1, col=2)

fig.update_xaxes(title_text='Log Return', row=1, col=1)
fig.update_yaxes(title_text='Density', row=1, col=1)
fig.update_xaxes(title_text='Theoretical Quantiles', row=1, col=2)
fig.update_yaxes(title_text='Sample Quantiles', row=1, col=2)
fig.update_layout(height=450, width=900, margin=dict(l=20,r=20,t=60,b=20))
fig.show()
print('The heavy tails suggest the returns are not exactly normal and come from different regimes with different volatilities')

The heavy tails suggest the returns are not exactly normal and come from different regimes with different volatilities


In order to fit our model we define a python object named `RegimeDetection`. We encapsulate a function to determine the market regime using the Gaussian Mixture Model, and an `initialise_model` function that acts as a flexible constructor, setting model attributes from a parameter dictionary.

In [8]:
class RegimeDetection:
    def get_regimes_gmm(self, input_data, params):
        gmm_model = self.initialise_model(GaussianMixture(), params).fit(input_data)
        return gmm_model

    def initialise_model(self, model, params):
        for parameter, value in params.items():
            setattr(model, parameter, value)
        return model


Furthermore we define a separate function to plot the component assignments returned by the GMM along with the log returns. Each colour represents a different Gaussian component (market regime).

In [9]:
def plot_gmm_regimes(component_labels, prices_df):
    '''
    Input:
    component_labels (numpy.ndarray) - array of GMM component assignments
    prices_df - Series of log returns

    Output:
    Graph showing component assignments and log returns
    '''
    colors = ['blue', 'green']
    n_components = len(np.unique(component_labels))
    fig = go.Figure()

    for i in range(n_components):
        mask = component_labels == i
        print(f'Number of observations for Component {i} : {len(prices_df.index[mask])}')

        fig.add_trace(go.Scatter(x=prices_df.index[mask], y=prices_df[mask],
                    mode='markers', name='Component ' + str(i),
                    marker=dict(size=4, color=colors[i])))
    fig.update_layout(height=400, width=900,
                      legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
                      margin=dict(l=20, r=20, t=20, b=20)).show()


After this we can then initialize the RegimeDetection object

In [10]:
regime_detection = RegimeDetection()

In [11]:
# making an array for the prices to be used in the model
price_array = np.array([[q] for q in apple['log_returns'].values])

In [12]:
params = {'n_components': 2, 'covariance_type': 'full', 'random_state': 100}
gmm_model = regime_detection.get_regimes_gmm(price_array, params)
gmm_states_raw = gmm_model.predict(price_array)
plot_gmm_regimes(np.array(gmm_states_raw), apple['log_returns'])

Number of observations for Component 0 : 1602
Number of observations for Component 1 : 407


In [13]:
# Extract and display GMM component parameters
means = gmm_model.means_.flatten()
vols  = np.sqrt(gmm_model.covariances_.flatten())
weights = gmm_model.weights_

# Sort so component 0 = bull (higher mean) and component 1 = bear (lower mean)
order = np.argsort(means)[::-1]
means   = means[order]
vols    = vols[order]
weights = weights[order]
labels  = ['Bull (low volatility)', 'Bear (high volatility)']

print('Estimated GMM Parameters')
print('='*50)
for i in range(2):
    print(f'\nComponent {i+1}: {labels[i]}')
    print(f'  Mixing weight:         {weights[i]:.4f}  ({weights[i]*100:.1f}% of trading days)')
    print(f'  Daily mean return:     {means[i]*100:.4f}%')
    print(f'  Annualised return:     {means[i]*252*100:.2f}%')
    print(f'  Daily volatility:      {vols[i]*100:.4f}%')
    print(f'  Annualised volatility: {vols[i]*np.sqrt(252)*100:.2f}%')

print('\nModel Selection Criteria')
print(f'  BIC: {gmm_model.bic(price_array):.2f}')
print(f'  AIC: {gmm_model.aic(price_array):.2f}')
print('  (Lower BIC/AIC = better fit; compare across models with different n_components)')

Estimated GMM Parameters

Component 1: Bull (low volatility)
  Mixing weight:         0.6401  (64.0% of trading days)
  Daily mean return:     0.2042%
  Annualised return:     51.45%
  Daily volatility:      1.0827%
  Annualised volatility: 17.19%

Component 2: Bear (high volatility)
  Mixing weight:         0.3599  (36.0% of trading days)
  Daily mean return:     -0.0987%
  Annualised return:     -24.86%
  Daily volatility:      2.8836%
  Annualised volatility: 45.78%

Model Selection Criteria
  BIC: -10453.35
  AIC: -10481.38
  (Lower BIC/AIC = better fit; compare across models with different n_components)


In [14]:
# Component assignments with posterior probabilities
remap = {order[0]: 0, order[1]: 1}
gmm_states_sorted = np.array([remap[s] for s in gmm_states_raw])

apple['Regime'] = gmm_states_sorted

posteriors = gmm_model.predict_proba(price_array)
apple['P_Bull'] = posteriors[:, order[0]]
apple['P_Bear'] = posteriors[:, order[1]]

print('Days assigned to each component:')
counts = apple['Regime'].value_counts().sort_index()
for i, lbl in enumerate(labels):
    pct = counts[i] / len(apple) * 100
    print(f'  {lbl}: {counts[i]} days ({pct:.1f}%)')

Days assigned to each component:
  Bull (low volatility): 1602 days (79.7%)
  Bear (high volatility): 407 days (20.3%)


In [15]:
# Regime-coloured plots: price, returns, posterior probabilities
close_series = apple_data['Close'].squeeze().loc[apple.index]

fig = make_subplots(rows=2, cols=1,
    subplot_titles=['AAPL Price — Coloured by Detected Regime',
                    'AAPL Log Returns — Coloured by Detected Regime',
                    ],
    shared_xaxes=True, vertical_spacing=0.07)

colors_regime = {0: 'green', 1: 'red'}
for regime in [0, 1]:
    mask = apple['Regime'] == regime
    fig.add_trace(go.Scatter(
        x=apple.index[mask], y=close_series[mask],
        mode='markers', marker=dict(size=3, color=colors_regime[regime]),
        name=labels[regime], legendgroup=labels[regime],
        showlegend=True), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=apple.index[mask], y=apple['log_returns'][mask],
        mode='markers', marker=dict(size=3, color=colors_regime[regime]),
        name=labels[regime], legendgroup=labels[regime],
        showlegend=False), row=2, col=1)



fig.add_hline(y=0, line_dash='dash', line_color='black', line_width=0.6, row=2, col=1)
fig.update_yaxes(title_text='Price (USD)', row=1, col=1)
fig.update_yaxes(title_text='Log Return', row=2, col=1)
fig.update_layout(height=900, width=900, margin=dict(l=20,r=20,t=60,b=20))
fig.show()

### Interpreting the Parameters

From the model output above:

- **Bull component:** The GMM identifies a Gaussian component with a positive daily mean return and relatively low volatility. The mixing weight tells us what fraction of all trading days are assigned to this component — typically the dominant one, reflecting the general upward drift of equity markets over this period.

- **Bear component:** This component has a near-zero or negative daily mean and much higher volatility. The COVID crash period (Feb–April 2020), the 2022 interest hikes and 2025 geopolitical issues are clearly captured here.

- **Mixing weights:** Unlike the HMM's transition matrix, the GMM does not model how long a regime persists or how likely it is to switch. A high bull mixing weight simply means that low-volatility days are more common overall — not that they cluster together. This is a key distinction when interpreting the results.

- **BIC and AIC:** These information criteria penalise model complexity. A 2-component model is a natural starting point, but if a 3-component model yields a substantially lower BIC, it may be worth adopting. Lower BIC/AIC always indicates a better-fitting model relative to its complexity.

---
## 5. Diagnosis — Checking the Model

In [16]:
# ADF test — check that log returns are stationary
adf_stat, p_val, lags, nobs, crit, _ = adfuller(apple['log_returns'], autolag='AIC')

print('ADF Test on AAPL Log Returns')
print(f'ADF statistic: {adf_stat:.4f}')
print(f'p-value:       {p_val:.6f}')
print(f'Critical values: {crit}')
print('')
if p_val < 0.05:
    print('Result: reject the unit root — log returns are stationary')
    print('This confirms the regime switching is in the variance, and not the mean/level of the regime')
else:
    print('Result: cannot reject unit root')

ADF Test on AAPL Log Returns
ADF statistic: -14.7011
p-value:       0.000000
Critical values: {'1%': np.float64(-3.433623856429125), '5%': np.float64(-2.862986213505), '10%': np.float64(-2.56753990225)}

Result: reject the unit root — log returns are stationary
This confirms the regime switching is in the variance, and not the mean/level of the regime


In [17]:
# Standardised residuals within each regime
apple['Std_Resid'] = np.nan
for k in range(2):
    mask = apple['Regime'] == k
    apple.loc[mask, 'Std_Resid'] = (apple.loc[mask, 'log_returns'] - means[k]) / vols[k]

resid = apple['Std_Resid'].dropna()
x_norm = np.linspace(-4, 4, 200)

# QQ of standardised residuals
(osm_r, osr_r), (slope_r, intercept_r, _) = stats.probplot(resid, dist='norm')
qq_rx = np.array([osm_r[0], osm_r[-1]])
qq_ry = intercept_r + slope_r * qq_rx

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['QQ Plot of Standardised Residuals',
                    'Standardised Residuals vs N(0,1)'])

fig.add_trace(go.Scatter(x=osm_r, y=osr_r, mode='markers',
    marker=dict(color='steelblue', size=3), name='Residuals'), row=1, col=1)
fig.add_trace(go.Scatter(x=qq_rx, y=qq_ry,
    line=dict(color='red'), name='QQ line'), row=1, col=1)

fig.add_trace(go.Histogram(x=resid, nbinsx=60, histnorm='probability density',
    marker_color='steelblue', opacity=0.6, name='Residuals'), row=1, col=2)
fig.add_trace(go.Scatter(x=x_norm, y=stats.norm.pdf(x_norm),
    line=dict(color='red', width=2), name='N(0,1)'), row=1, col=2)

fig.update_xaxes(title_text='Theoretical Quantiles', row=1, col=1)
fig.update_yaxes(title_text='Sample Quantiles', row=1, col=1)
fig.update_xaxes(title_text='Standardised Residual', row=1, col=2)
fig.update_yaxes(title_text='Density', row=1, col=2)
fig.update_layout(height=450, width=900, margin=dict(l=20,r=20,t=60,b=20))
fig.show()

In [18]:
# Skewness, kurtosis and Jarque-Bera test within each regime
print('Residual statistics within each regime:')
for k, lbl in enumerate(labels):
    resids = apple[apple['Regime'] == k]['Std_Resid'].dropna()
    print(f'  {lbl}:')
    print(f'    n = {len(resids)},  skewness = {resids.skew():.4f},  excess kurtosis = {resids.kurtosis():.4f}')
    jb, pval = stats.jarque_bera(resids)
    print(f'    Jarque-Bera: stat={jb:.2f}, p={pval:.4f}')

Residual statistics within each regime:
  Bull (low volatility):
    n = 1602,  skewness = 0.0993,  excess kurtosis = -0.7128
    Jarque-Bera: stat=36.69, p=0.0000
  Bear (high volatility):
    n = 407,  skewness = 0.2974,  excess kurtosis = 0.0805
    Jarque-Bera: stat=6.03, p=0.0491


---
## 6. Damage — Problems the Model Reveals

From the diagnostics above, we can see a few issues specific to the GMM approach:

1. **Temporal structure is completely ignored.** The most fundamental limitation of the GMM is that it treats each day's return as an independent draw. In reality, market regimes persist — a bear market typically lasts weeks or months. The GMM can assign consecutive days to opposite components if their returns happen to sit on different sides of the component boundary, which produces economically implausible day-to-day regime flipping.

2. **Heavy tails persist within components.** Even after separating the data into two Gaussian components, the standardised residuals still show excess kurtosis. The Gaussian assumption within each component is not perfectly met, suggesting that extreme return days are not fully explained by a two-component normal mixture.

3. **Number of components is chosen by hand.** While BIC guides the choice, selecting $K = 2$ is still a modelling decision. A third component (e.g. a recovery or transition state) might better capture the data without overfitting, and should be evaluated using BIC comparisons.

---
## 7. Directions — Can We Improve the Model?

Based on the diagnostics, here are some ways we could improve the model:

1. **Select the number of components using BIC.** Fit GMMs with $K = 2, 3, 4$ components and compare their BIC scores. The model with the lowest BIC offers the best trade-off between goodness-of-fit and complexity.

2. **Switch to a Hidden Markov Model to capture persistence.** If it is important to model the fact that regimes tend to persist (which it usually is in finance), an HMM adds a transition probability matrix that gives each regime temporal memory — something the GMM cannot provide.

3. **Try different covariance structures.** The `covariance_type` parameter can be set to `'tied'` (shared covariance across components) or `'diag'` (diagonal covariance) as lighter alternatives to `'full'`, which may reduce overfitting when data is limited.

4. **Use a rolling or online GMM.** Re-fitting the GMM on a rolling window of the most recent $N$ trading days would allow regime probabilities to adapt over time, making the model more suitable for live trading signals.

---
## 8. Deployment — How Would This Model Be Used?

In practice, a GMM-based regime detection model could be used in several ways:

1. **Volatility targeting:** The GMM provides a volatility estimate for each component. A portfolio manager could scale position sizes inversely with the posterior-weighted volatility — reducing exposure when $P(\text{bear})$ is high.

2. **Threshold-based signals:** The posterior probability $P(\text{bull})$ can serve as a continuous signal. When it falls below a threshold (e.g. 0.5), the portfolio shifts toward lower-risk assets such as bonds or cash.

3. **Feature engineering for ML models:** The GMM regime label or posterior probability can be added as a feature to a broader machine learning pipeline for return prediction or risk factor modelling.

4. **Retrospective regime labelling:** Because the GMM is fitted on the full sample, it is most reliable for historical analysis — identifying which periods correspond to which market conditions — rather than real-time signal generation. For live use, the model should be re-estimated periodically on a rolling basis.

In [19]:
# Simple regime-aware strategy: hold AAPL only when P(Bull) > 0.6
apple['Signal'] = (apple['P_Bull'] > 0.6).astype(int)
apple['Strategy'] = apple['Signal'].shift(1) * apple['log_returns']
apple['BuyHold']  = apple['log_returns']

apple['Cumul_Strategy'] = apple['Strategy'].fillna(0).cumsum().apply(np.exp)
apple['Cumul_BuyHold']  = apple['BuyHold'].cumsum().apply(np.exp)

def sharpe(r):
    return (r.mean() / r.std()) * np.sqrt(252)

print('Strategy performance (illustrative backtest only):')
print(f'  Regime-aware total return: {apple["Cumul_Strategy"].iloc[-1]-1:.2%}')
print(f'  Buy-and-hold total return: {apple["Cumul_BuyHold"].iloc[-1]-1:.2%}')
print(f'  Regime-aware Sharpe: {sharpe(apple["Strategy"].dropna()):.3f}')
print(f'  Buy-and-hold Sharpe: {sharpe(apple["BuyHold"].dropna()):.3f}')
print('')
print('Note: this backtest uses full-sample GMM component labels, so it has look-ahead bias.')
print('In a real deployment, the GMM would be re-estimated periodically using only past data.')

Strategy performance (illustrative backtest only):
  Regime-aware total return: 525.42%
  Buy-and-hold total return: 576.91%
  Regime-aware Sharpe: 0.975
  Buy-and-hold Sharpe: 0.780

Note: this backtest uses full-sample GMM component labels, so it has look-ahead bias.
In a real deployment, the GMM would be re-estimated periodically using only past data.


In [20]:
# Cumulative return plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=apple.index, y=apple['Cumul_BuyHold'],
    line=dict(color='blue'), name='Buy and Hold'))
fig.add_trace(go.Scatter(x=apple.index, y=apple['Cumul_Strategy'],
    line=dict(color='orange', dash='dash'), name='Regime-Aware Strategy'))
fig.update_layout(
    title='Cumulative Return: Regime-Aware Strategy vs Buy and Hold (AAPL 2018–2025)',
    yaxis_title='Growth of $1', xaxis_title='Date',
    height=450, width=900, margin=dict(l=20,r=20,t=60,b=20))
fig.show()

---
# Non-Technical Report

## Section 1 - What I Found

The analysis of Apple's daily stock returns from 2018 to 2025 using a Gaussian Mixture Model leads to the foreseeable conclusion that the market falls into 2 main regimes: a calm and stable component (more dominant over the years) and a turbulent component (which was less frequent).

- **Calm and Stable Component:** This was mainly shown in the steady, positive growth with minimal/small swings on a daily basis. This was the more frequent observed trend over the years.
- **Turbulent Component:** This was evidenced by the irregular and unpredictable price movement with large, significant daily swings. This component was less frequent (however very detrimental when it occurred).

## Section 2 - Recommended Course of Action

- When the market is in a turbulent unpredictable component it would be advisable to invest in more stable assets such as property and increasing reserves as well as a more conservative approach.
- When the calm component is more likely then add more positions on the Apple stock then maintain the old positions. The prolonged persistence of the calm market will result in rewards for continued investment in it.
- **As an ongoing discipline:** monitor the GMM posterior probabilities regularly and adjust exposure accordingly.
- Investment in AI is also advisable so as to have better prediction and monitoring systems.

## Section 3 - What Drove the Components

- **Calm component:** Mainly driven by the surge in consumer demand for Apple services, the increase in growth rate of the services, and the continued surgence in AI hype and optimism, coupled with breakthroughs in the field.
- **Turbulent component:** Mostly causes by the COVID-19 pandemic (systemic shocks), high inflation rates in 2022 and also the Russia invasion of Ukraine.
- For 2025 it can be attributed to geopolitical changes (such as the Trump Administration.)

The GMM is effective at identifying these structurally different periods even without modelling the temporal sequence of returns. Most of the changes can then be attributed to changes in macro-economic policies and geopolitics.

---
## References

- Dempster, A.P., Laird, N.M., & Rubin, D.B. (1977). Maximum Likelihood from Incomplete Data via the EM Algorithm. *Journal of the Royal Statistical Society: Series B*, 39(1), 1–38.

- McLachlan, G.J., & Peel, D. (2000). *Finite Mixture Models*. Wiley-Interscience.

- Hull, J.C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.

- scikit-learn GaussianMixture documentation: https://scikit-learn.org/stable/modules/mixture.html

- Yahoo Finance data accessed via yfinance Python library.